In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%run ../00_utils/common

In [0]:
%run ../00_utils/quality

In [0]:
BATCH = batch_id()

## trans_ventas

In [0]:
def process_ventas():
    print(" === Inicio trans_ventas ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.trans_ventas")

    df = (df
          # Deduplicar (Anomalía A1: transacciones duplicadas)
          .withColumn("_rn", F.row_number().over(Window.partitionBy("id_trans").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          # Campos obligatorios no nulos
          .filter(
              F.col("id_trans").isNotNull() &
              F.col("id_tienda").isNotNull() &
              F.col("art_id").isNotNull())
          .withColumn("fec_trans", F.to_date("fec_trans"))
          .withColumn("qty_vendida", F.col("qty_vendida").cast("integer"))
          .withColumn("precio_unitario_venta", F.col("precio_unitario_venta").cast("double"))
          .withColumn("descuento_aplicado", F.col("descuento_aplicado").cast("double"))
          # Aplicar id_miembro nulo -> cliente anónimo
          .withColumn("id_miembro",
                      F.when(F.col("id_miembro").isNull(), F.lit("ANONIMO"))
                      .otherwise(F.col("id_miembro")))
          .withColumn("_batch_id", F.lit(BATCH)))

    # Integridad referencial
    arts = spark.table(f"{CATALOG}.silver.mstr_articulos").select("art_id")
    tiendas = spark.table(f"{CATALOG}.silver.mstr_tiendas").select("id_tienda")

    orphan_art = check_referential_integrity(df, arts, "art_id", "art_id", "trans_ventas")
    if orphan_art.count():
        log_referential_errors(orphan_art, "trans_ventas", "art_id sin match en silver", BATCH)
    df = df.join(arts, on="art_id", how="inner")

    orphan_tda = check_referential_integrity(df, tiendas, "id_tienda", "id_tienda", "trans_ventas")
    if orphan_tda.count():
        log_referential_errors(orphan_tda, "trans_ventas", "id_tienda sin match en silver", BATCH)
    df = df.join(tiendas, on="id_tienda", how="inner")

    quality_report(df, "silver.trans_ventas", ["id_trans"], ["id_miembro"], ["qty_vendida", "precio_unitario_venta"])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.trans_ventas")
    log_run("silver", "trans_ventas", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.trans_ventas")


## inv_stock_diario

In [0]:
def process_stock():
    print(" === Inicio inv_stock_diario ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.inv_stock_diario")

    df = (df
          .withColumn("_rn", F.row_number().over(Window.partitionBy("id_snapshot").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          .filter(F.col("id_snapshot").isNotNull())
          .withColumn("fec_snapshot", F.to_date("fec_snapshot"))
          # Anomalía A3: stock_fisico negativo -> corregir a 0
          .withColumn("stock_fisico",
              F.when(F.col("stock_fisico") < 0, F.lit(0))
               .otherwise(F.col("stock_fisico").cast("integer")))
          # Aplicar nulos con 0
          .withColumn("stock_transito", F.coalesce(F.col("stock_transito").cast("integer"),  F.lit(0)))
          .withColumn("stock_reservado", F.coalesce(F.col("stock_reservado").cast("integer"), F.lit(0)))
          .withColumn("_batch_id", F.lit(BATCH)))

    quality_report(df, "silver.inv_stock_diario", ["id_snapshot"], [], ["stock_fisico", "stock_transito", "stock_reservado"])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.inv_stock_diario")
    log_run("silver", "inv_stock_diario", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.inv_stock_diario")


## process_devoluciones

In [0]:
def process_devoluciones():
    print(" === Inicio post_devoluciones ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.post_devoluciones")

    # Catálogo legible de motivos
    motivo_map = {
        "DEFECTO_FABRICA": "Defecto de fábrica",
        "PRODUCTO_DANADO": "Producto dañado",
        "TALLA_INCORRECTA": "Talla incorrecta",
        "CAMBIO_OPINION": "Cambio de opinión",
        "FECHA_VENCIDA": "Fecha vencida",
        "ERROR_DESPACHO": "Error de despacho",
        "OTRO": "Otro",
    }
    mapping_expr = F.create_map([F.lit(k) for pair in motivo_map.items() for k in pair])

    df = (df
          .withColumn("_rn", F.row_number().over(Window.partitionBy("id_devolucion").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          .filter(F.col("id_devolucion").isNotNull())
          .withColumn("fec_devolucion", F.to_date("fec_devolucion"))
          .withColumn("qty_devuelta", F.col("qty_devuelta").cast("integer"))
          .withColumn("vr_reembolso",
              F.coalesce(F.col("vr_reembolso").cast("double"), F.lit(0.0)))
          .withColumn("motivo_desc", mapping_expr[F.col("motivo_cod")])
          .withColumn("_batch_id", F.lit(BATCH)))

    # Anomalía A2: rechazar devoluciones con fecha anterior a la transacción
    ventas = spark.table(f"{CATALOG}.silver.trans_ventas").select("id_trans", "fec_trans")
    df = df.join(ventas, df.id_trans_origen == ventas.id_trans, "left")
    df_invalid = df.filter(F.col("fec_devolucion") < F.col("fec_trans"))
    if df_invalid.count():
        log_referential_errors(df_invalid, "post_devoluciones", "fec_devolucion anterior a fec_trans (A2)", BATCH)
    df = df.filter(
        F.col("fec_trans").isNull() | (F.col("fec_devolucion") >= F.col("fec_trans"))
        ).drop("fec_trans", "id_trans")

    quality_report(df, "silver.post_devoluciones", ["id_devolucion"], ["vr_reembolso"], ["qty_devuelta"])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.post_devoluciones")
    log_run("silver", "post_devoluciones", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.post_devoluciones")


## Execution

In [0]:

print(f"=== Silver Facts | batch={BATCH} ===")
try: process_ventas()
except Exception as e: print(f"  [ERROR] trans_ventas: {e}")
try: process_stock()
except Exception as e: print(f"  [ERROR] inv_stock_diario: {e}")
try: process_devoluciones()
except Exception as e: print(f"  [ERROR] post_devoluciones: {e}")
print("\n [DONE] Silver facts completado.")